# SSL + Context Domain Adaptation Runner

Run the crash-test experiment directly from this notebook. The training cell below uses the full schedule by default, not the 1-epoch smoke configuration.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
repo_root = cwd.parents[1] if cwd.name == 'self_supervised_domain_adaptation' else cwd
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from crash_tests.self_supervised_domain_adaptation.config import SSLDomainAdaptationConfig
from crash_tests.self_supervised_domain_adaptation.run_experiment import run_with_config


## Config

Edit the values in the next cell before running. The defaults below use the full training schedule from the crash-test config.

In [2]:
RUN_NAME = 'ssl_context_notebook_full'
RUN_STRESS_TEST = False
MAX_STRESS_STUDIES = None

SSL_PRETRAIN_EPOCHS = 30
SOURCE_WARMUP_EPOCHS = 3
SOURCE_TRAIN_EPOCHS = 18
FINETUNE_EPOCHS = 30

config = SSLDomainAdaptationConfig(
    parquet_path=repo_root / 'data/raw/33000_ROWS.parquet',
    output_root=repo_root / 'crash_tests' / 'self_supervised_domain_adaptation' / 'outputs',
    run_name=RUN_NAME,
    run_stress_test=RUN_STRESS_TEST,
    max_stress_studies=MAX_STRESS_STUDIES,
    ssl_pretrain_epochs=SSL_PRETRAIN_EPOCHS,
    source_warmup_epochs=SOURCE_WARMUP_EPOCHS,
    source_train_epochs=SOURCE_TRAIN_EPOCHS,
    finetune_epochs=FINETUNE_EPOCHS,
    verbose=True,
)

display(pd.DataFrame([config.to_dict()]))


,parquet_path,output_root,run_name,random_state,paired_final_holdout_rows,ssl_objective,targets,source_fmiss_target,run_unit_only_arm,run_contextual_arm,...,weight_decay,patience,lightgbm_estimators,lightgbm_learning_rate,lightgbm_num_leaves,save_checkpoints,run_stress_test,max_stress_studies,verbose,resolved_output_dir
0,/Users/paulruiz/Documents/Predicting_Good_Unit...,/Users/paulruiz/Documents/Predicting_Good_Unit...,ssl_context_notebook_full,42,100,masked_denoising_autoencoder,"(fpos, fmiss)",fmiss_extended,True,True,...,0.0001,5,250,0.05,31,True,False,None,True,/Users/paulruiz/Documents/Predicting_Good_Unit...


## Run Training

Execute the next cell to run the experiment. The cell output will stream the stage logs and epoch-by-epoch losses.

In [3]:
output_dir = run_with_config(config)
print(f'Saved outputs to {output_dir}')
output_dir


Loading parquet from /Users/paulruiz/Documents/Predicting_Good_Units/data/raw/33000_ROWS.parquet
main: SSL pretraining on 4963 unlabeled paired rows
ssl_pretrain: epochs=30 train_rows=2843 val_rows=2120
ssl_pretrain epoch=1/30 train_loss=1.880696 val_loss=1.414543
ssl_pretrain epoch=2/30 train_loss=1.291054 val_loss=1.184386
ssl_pretrain epoch=3/30 train_loss=1.163599 val_loss=1.023230
ssl_pretrain epoch=4/30 train_loss=1.036340 val_loss=0.954496
ssl_pretrain epoch=5/30 train_loss=0.978147 val_loss=0.975703
ssl_pretrain epoch=6/30 train_loss=1.061948 val_loss=0.966594
ssl_pretrain epoch=7/30 train_loss=0.921698 val_loss=0.851746
ssl_pretrain epoch=8/30 train_loss=0.899032 val_loss=0.859828
ssl_pretrain epoch=9/30 train_loss=0.900924 val_loss=0.798807
ssl_pretrain epoch=10/30 train_loss=1.055524 val_loss=0.814323
ssl_pretrain epoch=11/30 train_loss=1.582670 val_loss=0.766515
ssl_pretrain epoch=12/30 train_loss=0.850997 val_loss=0.858410
ssl_pretrain epoch=13/30 train_loss=0.750289 val_l

PosixPath('/Users/paulruiz/Documents/Predicting_Good_Units/crash_tests/self_supervised_domain_adaptation/outputs/ssl_context_notebook_full')

## Review Outputs

In [4]:
metrics = pd.read_csv(output_dir / 'metrics.csv')
baselines = pd.read_csv(output_dir / 'baseline_comparison.csv')
stress = pd.read_csv(output_dir / 'stress_test_results.csv')
history = pd.read_csv(output_dir / 'training_history.csv')
split_manifest = json.loads((output_dir / 'split_manifest.json').read_text())

display(pd.DataFrame([split_manifest['main']]))
display(metrics.sort_values(['target', 'mae', 'model_id']))


,allow_test_time_unlabeled_context,context_feature_count,context_feature_count_by_target,holdout_recordings,hybrid_source_rows,paired_finetune_recordings,paired_finetune_rows,paired_non_test_rows,paired_test_recordings,paired_test_rows,protocol,shape_feature_count,ssl_pool_recordings,ssl_pool_rows,training_side_scaler_rows,unit_feature_count
0,True,521,"{'fmiss': 62, 'fpos': 48}","[PAIRED_BOYDEN::paired_boyden32c::1103_1_1, PA...",24211,14,107,5070,15,100,recording_disjoint,70,16,4963,29281,164


,protocol,family,model_id,source_target,target,arm,selected_finetune_epochs,mae,rmse,r2,bias,calibration_slope,calibration_intercept
0,main,baseline,contextual_lightgbm_no_study_id,NaN,fmiss,NaN,NaN,0.186768,0.231416,0.361718,0.044327,1.044117,-0.058526
1,main,ssl,ssl_fmiss_contextual,fmiss_extended,fmiss,contextual,30.0,0.216543,0.283510,0.042006,-0.035802,0.589270,0.135084
2,main,ssl,ssl_fmiss_unit_only,fmiss_extended,fmiss,unit_only,30.0,0.222729,0.280502,0.062224,-0.024404,0.576991,0.131475
3,main,baseline,neural_no_ssl_fmiss_contextual,fmiss_extended,fmiss,contextual,15.0,0.243213,0.309615,-0.142541,-0.030616,0.359085,0.188862
4,main,baseline,neural_no_ssl_fmiss_unit_only,fmiss_extended,fmiss,unit_only,30.0,0.250164,0.316840,-0.196481,-0.037847,0.253259,0.216822
5,main,baseline,paired_only_lightgbm_shape,NaN,fmiss,NaN,NaN,0.260319,0.317302,-0.199978,0.022013,0.124732,0.240160
6,main,baseline,neural_no_ssl_fpos_unit_only,fpos,fpos,unit_only,12.0,0.135777,0.211771,0.261492,-0.070137,0.857914,0.091560
7,main,baseline,contextual_lightgbm_no_study_id,NaN,fpos,NaN,NaN,0.153214,0.196654,0.363164,0.022210,0.898287,0.002518
8,main,baseline,paired_only_lightgbm_shape,NaN,fpos,NaN,NaN,0.159735,0.202304,0.326047,0.036435,0.864874,-0.001661
9,main,baseline,neural_no_ssl_fpos_contextual,fpos,fpos,contextual,30.0,0.199200,0.263923,-0.147032,0.015653,0.395878,0.127260


In [11]:
print('Baseline comparison')
display(baselines.sort_values(['target', 'mae', 'model_id']))

print('Training history tail')
display(history.tail(50))

if len(stress):
    print('Stress test results')
    display(stress.sort_values(['held_out_study', 'target', 'mae', 'model_id']))
else:
    print('Stress test disabled or no rows produced for this run.')


Baseline comparison


,protocol,family,model_id,source_target,target,arm,selected_finetune_epochs,mae,rmse,r2,bias,calibration_slope,calibration_intercept
0,main,baseline,contextual_lightgbm_no_study_id,NaN,fmiss,NaN,NaN,0.186768,0.231416,0.361718,0.044327,1.044117,-0.058526
1,main,baseline,neural_no_ssl_fmiss_unit_only,fmiss_extended,fmiss,unit_only,2.0,0.229454,0.325932,-0.266138,-0.162247,0.649132,0.202694
2,main,baseline,paired_only_lightgbm_shape,NaN,fmiss,NaN,NaN,0.260319,0.317302,-0.199978,0.022013,0.124732,0.240160
3,main,baseline,neural_no_ssl_fmiss_contextual,fmiss_extended,fmiss,contextual,6.0,0.266268,0.315884,-0.189279,0.060329,0.323577,0.168201
4,main,baseline,contextual_lightgbm_no_study_id,NaN,fpos,NaN,NaN,0.153214,0.196654,0.363164,0.022210,0.898287,0.002518
5,main,baseline,paired_only_lightgbm_shape,NaN,fpos,NaN,NaN,0.159735,0.202304,0.326047,0.036435,0.864874,-0.001661
6,main,baseline,neural_no_ssl_fpos_contextual,fpos,fpos,contextual,16.0,0.200658,0.260679,-0.119007,0.061700,0.445737,0.094940
7,main,baseline,neural_no_ssl_fpos_unit_only,fpos,fpos,unit_only,4.0,0.200693,0.287796,-0.363927,-0.104193,0.228538,0.194236


Training history tail


,protocol,model_id,stage,split_id,epoch,train_loss,val_loss,val_mae
243,main,ssl_fmiss_contextual,source,NaN,12,1.715162,NaN,0.060824
244,main,ssl_fmiss_contextual,source,NaN,13,1.621736,NaN,0.053886
245,main,ssl_fmiss_contextual,source,NaN,14,1.532308,NaN,0.051683
246,main,ssl_fmiss_contextual,source,NaN,15,1.476940,NaN,0.064327
247,main,ssl_fmiss_contextual,source,NaN,16,1.443924,NaN,0.052604
248,main,ssl_fmiss_contextual,source,NaN,17,1.364850,NaN,0.054011
249,main,ssl_fmiss_contextual,source,NaN,18,1.323883,NaN,0.050971
250,main,ssl_fmiss_contextual,paired_cv,1.0,1,288.630646,NaN,0.320726
251,main,ssl_fmiss_contextual,paired_cv,1.0,2,49.037796,NaN,0.336002
252,main,ssl_fmiss_contextual,paired_cv,1.0,3,26.917011,NaN,0.310089


Stress test disabled or no rows produced for this run.
